# Задача 1: Подбор олигонуклеотидов с заданной аффинностью

## Условие:
Дана олигонуклеотидная последовательность-мишень. Необходимо подобрать олигонуклеотид, который будет связываться с мишенью с аффинностью, попадающей в заданный диапазон.


In [1]:
# 1. ПОДГОТОВКА ОКРУЖЕНИЯ
!wget https://nikitinlab.ru/hackathon/intensiv/predictors_ext.py
!gdown 1NW6VKxIH2rk4Clk4ioprrLEUMkoS0gXT
!unzip -qq nupack-4.0.1.12.zip
!pip install -U nupack -f nupack-4.0.1.12/package

--2025-09-08 16:53:02--  https://nikitinlab.ru/hackathon/intensiv/predictors_ext.py
Resolving nikitinlab.ru (nikitinlab.ru)... 37.140.192.24, 2a00:f940:2:2:1:1:0:41
Connecting to nikitinlab.ru (nikitinlab.ru)|37.140.192.24|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5200 (5.1K)
Saving to: ‘predictors_ext.py’

predictors_ext.py   100%[===================>]   5.08K  --.-KB/s    in 0s      

2025-09-08 16:53:02 (1.03 GB/s) - ‘predictors_ext.py’ saved [5200/5200]

Downloading...
From (original): https://drive.google.com/uc?id=1NW6VKxIH2rk4Clk4ioprrLEUMkoS0gXT
From (redirected): https://drive.google.com/uc?id=1NW6VKxIH2rk4Clk4ioprrLEUMkoS0gXT&confirm=t&uuid=450096cf-6227-4955-bee7-de0ea242f372
To: /content/nupack-4.0.1.12.zip
100% 240M/240M [00:04<00:00, 52.4MB/s]
Looking in links: nupack-4.0.1.12/package
Processing ./nupack-4.0.1.12/package/nupack-4.0.1.12-cp312-cp312-linux_x86_64.whl


In [2]:
from predictors_ext import *
import pandas as pd

In [3]:
#Олигонуклеотиды-мишени и границы аффинности для них. Для каждого из них необходимо подобрать новый олиг той же длины, связывающийся с Kd_min < Kd < Kd_max
!gdown 1dLvWT_gArh0DBonHtgdsAIr8Gn-z8kC0
df = pd.read_excel("Task1_simple_aff_check_data.xlsx")
df

Downloading...
From: https://drive.google.com/uc?id=1dLvWT_gArh0DBonHtgdsAIr8Gn-z8kC0
To: /content/Task1_simple_aff_check_data.xlsx
100% 17.1k/17.1k [00:00<00:00, 29.6MB/s]


,seq,Kd_min,Kd_max
0,GGACAATAAG,1.000000e-08,1.000000e-09
1,CGAGAAGCTTGG,1.000000e-08,1.000000e-09
2,CACGACTTCCATTC,1.000000e-05,1.000000e-06
3,ATTTCGACTGGACAAT,1.000000e-06,1.000000e-07
4,ACCAAATTAAAGCGTCAT,1.000000e-11,1.000000e-12


***Ошибка в значениях, числа Kd_min больше чем Kd_max, соответственно никаких чисел между Kd_min и Kd_max как в условии задачи быть не может (Kd_min < Kd < Kd_max --> None). Я решала с поправкой: Kd_min > Kd > Kd_max***

###Решение

In [4]:
nupack_pred = Nupack_Affinity_Predictor()
hairpin_pred = Nupack_Hairpin_Predictor()

####Напишем функцию комплементарной цепочки ДНК.

*P.S. Чуть позже я нашла аналогичную в predictors_ext.py, но оставила свою.*

In [5]:
#get reverse complement sequence
def reverse_complement(seq):
    compl_rules = {'A':'T', 'G':'C', 'C':'G', 'T':'A'}
    rev_compl_seq = ''
    for b in seq:
        rev_compl_seq += compl_rules.get(b)
    return rev_compl_seq[::-1]

####Получим для каждого комплементарный олигонуклеотид и добавим в таблицу

In [6]:
#for each seq get reverse complement seq
compl_oligs = []
for olig_target in list(df['seq']):
    compl_oligs.append(reverse_complement(olig_target))

df['complement oligs'] = compl_oligs
df

,seq,Kd_min,Kd_max,complement oligs
0,GGACAATAAG,1.000000e-08,1.000000e-09,CTTATTGTCC
1,CGAGAAGCTTGG,1.000000e-08,1.000000e-09,CCAAGCTTCTCG
2,CACGACTTCCATTC,1.000000e-05,1.000000e-06,GAATGGAAGTCGTG
3,ATTTCGACTGGACAAT,1.000000e-06,1.000000e-07,ATTGTCCAGTCGAAAT
4,ACCAAATTAAAGCGTCAT,1.000000e-11,1.000000e-12,ATGACGCTTTAATTTGGT


####Проверим их Kd и занесем в таблицу

In [7]:
#check their Kd
current_kd = []
for pair_number in range(len(df)):
    current_kd.append(float(nupack_pred.predict([df['seq'][pair_number]], [df['complement oligs'][pair_number]])))
df['Kd'] = current_kd
df

/tmp/ipython-input-3513820204.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  current_kd.append(float(nupack_pred.predict([df['seq'][pair_number]], [df['complement oligs'][pair_number]])))


,seq,Kd_min,Kd_max,complement oligs,Kd
0,GGACAATAAG,1.000000e-08,1.000000e-09,CTTATTGTCC,4.314158e-09
1,CGAGAAGCTTGG,1.000000e-08,1.000000e-09,CCAAGCTTCTCG,7.191674e-13
2,CACGACTTCCATTC,1.000000e-05,1.000000e-06,GAATGGAAGTCGTG,1.154358e-14
3,ATTTCGACTGGACAAT,1.000000e-06,1.000000e-07,ATTGTCCAGTCGAAAT,5.193202e-16
4,ACCAAATTAAAGCGTCAT,1.000000e-11,1.000000e-12,ATGACGCTTTAATTTGGT,8.459647e-19


####Напишем функцию, изменяющую нуклеотид(ы) в комлементарном олигонуклеотиде

*P.S. Чуть позже я нашла аналогичную в predictors_ext.py, но оставила свою.*

In [8]:
#mutate n_bases in seq
def olig_mutation(seq, n_bases):
    base_positions = random.sample(range(len(seq)), n_bases)
    for base_pos in base_positions:
        seq = seq[:base_pos] + random.choice('ATGC'.replace(seq[base_pos], '')) + seq[base_pos+1:]
    return seq

####Попробуем рандомно заменять нуклеотиды c целью получить олигонуклеотиды с нужной аффинностью. Однако нужно учесть возможность образования шпилек, поэтому дополнительно делаем проверку.

In [9]:
final_oligs = []
final_kd = []
final_hairpin_pred = []

for number in range(len(df)):
    n_bases = 1
    try_count = 0 #number of mutation tries (for one number of n_bases)
    while n_bases != len(df['complement oligs'][number]):
        seq = olig_mutation(df['complement oligs'][number], n_bases) #mutate try
        current_kd = float(nupack_pred.predict([df['seq'][number]], [seq]))
        if current_kd > df['Kd_max'][number] and current_kd < df['Kd_min'][number]: #check Kd_min > Kd > Kd_max
            current_hairpin_pred = hairpin_pred.predict([seq])
            if float(current_hairpin_pred) == 0.0: #check hairpin prediction
                final_oligs.append(seq)
                final_kd.append(current_kd)
                final_hairpin_pred.append(float(current_hairpin_pred))
                break
            else:
                try_count += 1
                continue
        else:
            if try_count > 100:
                n_bases += 1
                try_count = 0
                continue
            else:
                try_count += 1
                continue
df['Final oligonucleotides'] = final_oligs
df['Final kd'] = final_kd
df['Hairpin prediction'] = final_hairpin_pred
df

/tmp/ipython-input-1871210544.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  current_kd = float(nupack_pred.predict([df['seq'][number]], [seq]))
/tmp/ipython-input-1871210544.py:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if float(current_hairpin_pred) == 0.0: #check hairpin prediction
/tmp/ipython-input-1871210544.py:16: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  final_hairpin_pred.append(float(current_hairpin_pred))


,seq,Kd_min,Kd_max,complement oligs,Kd,Final oligonucleotides,Final kd,Hairpin prediction
0,GGACAATAAG,1.000000e-08,1.000000e-09,CTTATTGTCC,4.314158e-09,CTTATTGTCA,7.320494e-09,0.0
1,CGAGAAGCTTGG,1.000000e-08,1.000000e-09,CCAAGCTTCTCG,7.191674e-13,CCAAGCTCCTCG,3.630410e-09,0.0
2,CACGACTTCCATTC,1.000000e-05,1.000000e-06,GAATGGAAGTCGTG,1.154358e-14,GAATGTAAGTTGTG,1.991746e-06,0.0
3,ATTTCGACTGGACAAT,1.000000e-06,1.000000e-07,ATTGTCCAGTCGAAAT,5.193202e-16,ATTGCCCACTCGAACT,3.182084e-07,0.0
4,ACCAAATTAAAGCGTCAT,1.000000e-11,1.000000e-12,ATGACGCTTTAATTTGGT,8.459647e-19,ATGACGCTCTATTTTGGT,1.321639e-12,0.0


In [10]:
# Отправка решения на проверку
# Измените на ваш идентификатор
participant_id = "Григорьева Екатерина Вячеславовна"
#Внесите пять подобранных последовательностей в этот список
final_results = ['CTTATTGTCA', 'CCAAGCTCCTCG', 'GAATGTAAGTTGTG', 'ATTGCCCACTCGAACT', 'ATGACGCTCTATTTTGGT']


import requests
import numpy as np
import random

# Функция для выполнения запроса к API
def submit_task1(participant_id, seq_arr):
    api_url = "http://87.242.73.209:6080/api/hackaton/task1"

    print('seq_arr', seq_arr)
    data = {
        "participant_id": participant_id,
        "seq_arr": seq_arr
    }

    try:
        response = requests.post(api_url, json=data)
        print(f"Статус код: {response.status_code}")

        if response.status_code == 200:
            result = response.json()
            print(f"Результат проверки: {result['result']}")
            if result.get('details'):
                print(f"Детали: {result['details']}")
        else:
            print(f"Ошибка: {response.text}")
    except Exception as e:
        print(f"Ошибка при отправке запроса: {str(e)}")



submit_task1(participant_id, final_results)

seq_arr ['CTTATTGTCA', 'CCAAGCTCCTCG', 'GAATGTAAGTTGTG', 'ATTGCCCACTCGAACT', 'ATGACGCTCTATTTTGGT']
Статус код: 200
Результат проверки: PASSED
